<!-- DCA Portfolio Relationship Notebook v3 -->
# VFX Production Intelligence Dashboard — Validate Table Relationships

Use the **Visual Guide** in Data Career Accelerator for table schemas, key candidates,
and relationship details. This notebook is only for running the checks, reviewing the
results, and recording what you learned.


## Start here

Run the collapsed setup cell once. It prepares an isolated DuckDB session from the
project's configured raw files, so it will not conflict with the DuckDB database open
in VS Code.


In [2]:
%pip install --upgrade jupysql duckdb pyyaml ipykernel

  Using cached jupysql-0.11.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached prettytable-3.18.0-py3-none-any.whl.metadata (37 kB)
  Using cached sqlalchemy-2.0.51-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7 kB)
  Using cached ipython_genutils-0.2.0-py2.py3-none-any.whl.metadata (755 bytes)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached sqlglot-30.13.0-py3-none-any.whl.metadata (24 kB)
  Using cached jupysql_plugin-0.4.5-py3-none-any.whl.metadata (7.8 kB)
  Using cached ploomber_core-0.2.27-py3-none-any.whl.metadata (532 bytes)
  Using cached posthog-7.27.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached charset_normalizer-3.4.9-cp3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
%load_ext sql
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

from pathlib import Path
import duckdb
import yaml

cwd = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (cwd, *cwd.parents, cwd / "projects" / "project-01-vfx-production-intelligence")
        if (candidate / "config" / "project_sources.yaml").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Open this project's generated VS Code workspace, then run the setup cell again."
    )

config = yaml.safe_load(
    (project_root / "config" / "project_sources.yaml").read_text(encoding="utf-8")
) or {}
schema_name = str(config.get("schema") or "raw")

def qid(value):
    return '"' + str(value).replace('"', '""') + '"'

def qpath(value):
    return "'" + Path(value).resolve().as_posix().replace("'", "''") + "'"

def reader(path, source_format):
    source_path = qpath(path)
    if source_format == "csv":
        return f"read_csv_auto({source_path}, header = true)"
    if source_format == "parquet":
        return f"read_parquet({source_path})"
    if source_format == "json_lines":
        return f"read_json_auto({source_path}, format = 'newline_delimited')"
    if source_format == "json":
        return f"read_json_auto({source_path})"
    raise ValueError(f"Unsupported project source format: {source_format}")

con = duckdb.connect(":memory:")
con.execute(f"CREATE SCHEMA IF NOT EXISTS {qid(schema_name)}")
for source in config.get("sources") or []:
    source_path = project_root / source["path"]
    if not source_path.is_file():
        raise FileNotFoundError(f"Project source file is missing: {source_path}")
    con.execute(
        f"CREATE OR REPLACE VIEW {qid(schema_name)}.{qid(source['view'])} AS "
        f"SELECT * FROM {reader(source_path, source['format'])}"
    )

%sql con --alias project
print(f"Project data ready: {len(config.get('sources') or [])} tables loaded.")


Project data ready: 6 tables loaded.


# 1. Check primary-key uniqueness

Using the key candidates listed in the Visual Guide, write the checks needed to find
identifiers that appear more than once. Add another SQL cell if you want to keep each
table's check separate.


In [13]:
%%sql

SELECT artist_id, COUNT(*)
FROM raw.artists
GROUP BY artist_id
HAVING COUNT(*) > 1;


artist_id,count_star()
ART-044,2
ART-053,2


In [12]:
%%sql

SELECT client_id, COUNT(*)
FROM raw.clients
GROUP BY client_id
HAVING COUNT(*) > 1;

client_id,count_star()
CL-012,2
CL-007,2


In [19]:
%%sql

SELECT project_id, COUNT(*)
FROM raw.projects
GROUP BY project_id
HAVING COUNT(*) > 1;

project_id,count_star()
PRJ-1005,2


### What did the uniqueness checks show?

Record any duplicated identifiers, explain whether they are expected, and note what
must be cleaned or documented before those keys are used in joins.

_Write your findings here._


# 2. Check for missing parent records

Use the relationships listed in the Visual Guide. Write `LEFT JOIN` checks that keep
all child records and identify foreign-key values that do not match a parent record.


In [ ]:
%%sql

-- Write your missing-parent checks here.


### What did the missing-parent checks show?

Record any orphaned records, describe the likely cause, and explain whether the issue
needs cleaning, exclusion, or a documented exception.

_Write your findings here._


# 3. Check join cardinality

Compare child-table row counts before and after the relationships you plan to use.
Investigate any join that unexpectedly multiplies or removes rows.


In [ ]:
%%sql

-- Write your join-cardinality checks here.


### What did the row-count comparisons show?

State which joins preserved the expected grain and explain any unexpected increases or
decreases in row counts.

_Write your findings here._


# 4. Add project-specific consistency checks

Review the project brief and data dictionary. Add any additional check needed for a
relationship rule that cannot be inferred from column names alone.


In [ ]:
%%sql

-- Write any project-specific consistency checks here.


### What did the additional checks show?

Describe the business rule you tested, why it matters, and what action the result
suggests.

_Write your findings here._


# Final relationship-validation conclusion

## Duplicate primary keys

_Summarize the uniqueness results._

## Orphaned foreign keys

_Summarize the missing-parent results._

## Join cardinality

_Summarize whether the tested joins preserved the expected row counts._

## Exceptions and cleanup decisions

_Document intentional exceptions, source-data problems, and required cleanup._

## Final decision

_State which relationships are safe to use, which are not, and what must happen before analysis continues._

## Completion checklist

- [ ] Every primary-key candidate needed by the analysis was checked.
- [ ] Every required relationship was checked for missing parent records.
- [ ] Join cardinality was checked for the relationships used in the analysis.
- [ ] Project-specific consistency rules were considered.
- [ ] Each result has a written interpretation.
- [ ] The notebook runs from top to bottom in a fresh kernel.
